# Estimators

## step 1: Mapping the problem

In [ ]:
from qiskit.circuit import Parameter
rx_angle = Parameter("rx_angle")
trotter_steps =2
qc = generate_1d_tfim_circuit(num_qubits, trotter_steps, rx_angle)

from qiskit.quantum_info import SparsePauliOp

middle_index = num_qubits // 2
observable = SparsePauliOp("I" * middle_index + "Z" + "I" * ( middle_index - 1))

## Step 2: Optimize the circuit

In [ ]:
from qiskit import transpile
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
 
service = QiskitRuntimeService()
 
backend = service.least_busy(
    simulator=False, operational=True, min_num_qubits=100
)
# pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuit = pm.run(qc)
qc_transpiled = transpile(qc, backend=backend, optimization_level=1)
observable = observable.apply_layout(qc_transpiled.layout)

## Step 3: Execute on quantum hardware

In [ ]:
from qiskit_ibm_runtime import EstimatorOptions
from qiskit_ibm_runtime import EstimatorV2 as Estimator

min_rx_angle = 0
max_rx_angle = np.pi / 2
num_rx_angles = 12
rx_angle_list = np.linspace(min_rx_angle, max_rx_angle, num_rx_angles)
 
options = EstimatorOptions()
options.resilience_level = 1
options.dynamical_decoupling.enable = True
options.dynamical_decoupling.sequence_type = "XY4"
 
# Create an Estimator object
estimator = Estimator(backend, options=options)

In [ ]:
# Submit the circuit to Sampler
job = estimator.run([(qc_transpiled, observable, rx_angle_list)])
job_id = job.job_id()
print(job_id)

## Step 4: Post-processing and plotting

In [ ]:
job = service.job(job_id)

exp_val_list = job.result()[0].data.evs

In [ ]:
import matplotlib.pyplot as plt
plt.plot(rx_angle_list / np.pi, exp_val_list, '--o')
# plt.plot(rx_angle_list / np.pi/2, exp_val_list, '--o')
plt.xlabel(r'RX Angle ( $\pi$)')
plt.ylabel(r'$\langle Z angle$ on Middle Qubit')
plt.ylim(-0.1,1.1)